In [1]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from google import genai
from google.genai import types as gtypes

try:
    from dotenv import load_dotenv

    load_dotenv()
except ModuleNotFoundError:
    print("python-dotenv is not installed. Run: pip install python-dotenv")

# =========================
# 0) CONFIG
# =========================
SUBJECT = os.getenv("SUBJECT", "Kimia").strip().title()
SUBJECT_SLUG = os.getenv("SUBJECT_SLUG", SUBJECT.lower()).strip().lower()
GRADE = os.getenv("GRADE", "XII").strip().upper()

BASE_DIR = Path(os.getenv("CONSENSUS_BASE_DIR", Path.cwd())).expanduser()
NOTEBOOK_DIR_FALLBACK = Path(os.getenv("CONSENSUS_NOTEBOOK_DIR", BASE_DIR / "final-consensus")).expanduser()
VALIDATION_DIR = BASE_DIR / "validations"
KG_DIR = BASE_DIR / "kg"
EBOOK_CONTEXT_DIR = BASE_DIR / "pdf-extracted"
CHECKPOINT_DIR = VALIDATION_DIR.parent / "checkpoints"

# Fallbacks for notebook execution from another working directory.
if not VALIDATION_DIR.exists():
    VALIDATION_DIR = NOTEBOOK_DIR_FALLBACK / "validations"
if not KG_DIR.exists():
    KG_DIR = NOTEBOOK_DIR_FALLBACK / "kg"
if not EBOOK_CONTEXT_DIR.exists():
    EBOOK_CONTEXT_DIR = NOTEBOOK_DIR_FALLBACK.parent / "pdf-extracted"

MASTER_KG_PATH = KG_DIR / f"{SUBJECT} Kelas {GRADE}.json"
R1_PATH = VALIDATION_DIR / f"expert-{SUBJECT_SLUG}-4.json"
R2_PATH = VALIDATION_DIR / f"expert-{SUBJECT_SLUG}-6.json"
OUTPUT_PATH = VALIDATION_DIR.parent / f"{SUBJECT_SLUG}_gold_standard.json"
OUTPUT_KG_PATH = KG_DIR / f"{SUBJECT} Kelas {GRADE}.consensus.json"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def display_path(path: Path) -> str:
    try:
        return str(path.relative_to(BASE_DIR))
    except ValueError:
        return path.name

required_paths = {
    "reviewer 1 validation": R1_PATH,
    "reviewer 2 validation": R2_PATH,
    "master KG": MASTER_KG_PATH,
}
missing_paths = [f"{label}: {display_path(path)}" for label, path in required_paths.items() if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required input files:\n" + "\n".join(missing_paths))

existing_outputs = [path for path in [OUTPUT_PATH, OUTPUT_KG_PATH] if path.exists()]
if existing_outputs:
    existing_list = "\n".join(display_path(path) for path in existing_outputs)
    raise FileExistsError(
        "Existing output file(s) found for this course. Delete or rename them before rerunning "
        "to avoid spending LLM tokens on a duplicate consensus run:\n"
        f"{existing_list}"
    )

print(f"Using subject: {SUBJECT} Kelas {GRADE}")
print(f"Using validation directory: {display_path(VALIDATION_DIR)}")
print(f"Using reviewer files: {R1_PATH.name}, {R2_PATH.name}")
print(f"Using master KG file: {display_path(MASTER_KG_PATH)}")
print(f"Using ebook context directory: {display_path(EBOOK_CONTEXT_DIR)}")
print(f"Using checkpoint directory: {display_path(CHECKPOINT_DIR)}")
print(f"Gold standard output will be written to: {display_path(OUTPUT_PATH)}")
print(f"Consensus KG output will be written to: {display_path(OUTPUT_KG_PATH)}")


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def normalize_label(label: str) -> str:
    if label is None:
        return "unknown"
    norm = str(label).strip().lower()
    alias_map = {
        "ignore": "abaikan",
        "ignored": "abaikan",
        "skip": "abaikan",
        "skipped": "abaikan",
    }
    norm = alias_map.get(norm, norm)

    allowed = {"correct", "wrong", "partial", "missing", "abaikan"}
    if norm in allowed:
        return norm
    return "unknown"


def coerce_triple_id(value):
    if value is None:
        return value
    if isinstance(value, int):
        return value
    text = str(value).strip()
    return int(text) if text.isdigit() else text


def _content_key(subject, relation, target):
    """Stable join key for a triple: (source concept, relation type, target)."""
    norm = lambda s: re.sub(r"\s+", " ", str(s or "").strip().lower())
    return (norm(subject), norm(relation), norm(target))

def _iter_concept_relations(kg_json_data):
    """Canonical triple walk: chapter -> subtopic -> concept -> relation.

    0-based and concept-relations only, matching the order/indexing the review
    app used to assign reviewer rating ids. chapter_relations are intentionally
    NOT counted here (the write-back walk excludes them too).
    """
    idx = 0
    for chapter_node in kg_json_data.get("chapters", []):
        chapter_name = chapter_node.get("chapter", "")
        for subtopic_node in chapter_node.get("subtopics", []):
            for concept_node in subtopic_node.get("concepts", []):
                subject_name = concept_node.get("name", "")
                for relation_node in concept_node.get("relations", []):
                    yield idx, chapter_name, subject_name, relation_node
                    idx += 1


def get_triple_context_by_id(kg_json_data, target_id):
    target_id = coerce_triple_id(target_id)
    for idx, chapter_name, subject_name, relation_node in _iter_concept_relations(kg_json_data):
        if idx == target_id:
            return {
                "chapter": chapter_name,
                "subject": subject_name,
                "relation": relation_node.get("type", ""),
                "target": relation_node.get("target_concept", relation_node.get("target", "")),
                "description": relation_node.get("description", ""),
            }
    return {"chapter": None, "subject": None, "relation": None, "target": None, "description": None}




Using subject: Kimia Kelas XII
Using validation directory: validations
Using reviewer files: expert-kimia-4.json, expert-kimia-6.json
Using master KG file: kg/Kimia Kelas XII.json
Using ebook context directory: pdf-extracted
Using checkpoint directory: checkpoints
Gold standard output will be written to: kimia_gold_standard.json
Consensus KG output will be written to: kg/Kimia Kelas XII.consensus.json


In [2]:
# =========================
# 1) LOAD & PARSE DATA
# =========================
data_r1 = load_json(R1_PATH)
data_r2 = load_json(R2_PATH)
kg_json_data = load_json(MASTER_KG_PATH)

ratings_r1 = data_r1.get("ratings", {})
ratings_r2 = data_r2.get("ratings", {})

all_triple_ids = sorted(
    set(ratings_r1.keys()) | set(ratings_r2.keys()),
    key=lambda x: int(x) if str(x).isdigit() else str(x),
)

df_ratings = pd.DataFrame({"triple_id_raw": all_triple_ids})
df_ratings["triple_id"] = df_ratings["triple_id_raw"].apply(coerce_triple_id)
df_ratings["r1_rating"] = df_ratings["triple_id_raw"].map(ratings_r1).apply(normalize_label)
df_ratings["r2_rating"] = df_ratings["triple_id_raw"].map(ratings_r2).apply(normalize_label)

context_rows = df_ratings["triple_id"].apply(lambda x: get_triple_context_by_id(kg_json_data, x))
context_df = pd.json_normalize(context_rows)
df_ratings = pd.concat([df_ratings, context_df], axis=1)

# Keep the pipeline context-complete by excluding unmapped rows from the main consensus flow.
df_unmapped_ratings = df_ratings[df_ratings["chapter"].isna()].copy()
df_ratings = df_ratings[df_ratings["chapter"].notna()].copy().reset_index(drop=True)

if not df_unmapped_ratings.empty:
    print("Unmapped ratings rows excluded from main flow:", len(df_unmapped_ratings))

print("df_ratings shape:", df_ratings.shape)
display(df_ratings.head())

df_ratings shape: (166, 9)


,triple_id_raw,triple_id,r1_rating,r2_rating,chapter,subject,relation,target,description
0,0,0,correct,correct,LARUTAN DAN KOLOID,Asam Brønsted-Lowry,MEMILIKI_SIFAT,Donor Proton,Asam Brønsted-Lowry dicirikan oleh kemampuanny...
1,1,1,correct,correct,LARUTAN DAN KOLOID,Basa Brønsted-Lowry,MEMILIKI_SIFAT,Akseptor Proton,Basa Brønsted-Lowry dicirikan oleh kemampuanny...
2,2,2,correct,correct,LARUTAN DAN KOLOID,Asam Lewis,MEMILIKI_SIFAT,Akseptor Pasangan Elektron Bebas,Asam Lewis berperan sebagai penerima pasangan ...
3,3,3,correct,correct,LARUTAN DAN KOLOID,Basa Lewis,MEMILIKI_SIFAT,Donor Pasangan Elektron Bebas,Basa Lewis berperan sebagai pemberi pasangan e...
4,4,4,correct,missing,LARUTAN DAN KOLOID,pH,DIFORMULASIKAN_SEBAGAI,–log [H+],pH dihitung dari nilai negatif logaritma konse...


In [3]:
# =========================
# 2) SPLIT AGREED VS DISAGREED
# =========================
df_agreed = df_ratings[df_ratings["r1_rating"] == df_ratings["r2_rating"]].copy()
df_agreed["final_consensus_label"] = df_agreed["r1_rating"]
df_agreed["reasoning"] = "Auto-agreed: both reviewers gave the same label."
df_agreed["source"] = "system_generated"
df_agreed["bucket"] = "ratings_agreed"

df_disagreed = df_ratings[df_ratings["r1_rating"] != df_ratings["r2_rating"]].copy()

print("df_agreed shape:", df_agreed.shape)
print("df_disagreed shape:", df_disagreed.shape)

df_agreed shape: (108, 13)
df_disagreed shape: (58, 9)


In [4]:
# =========================
# 3) HANDLE MANUALLY ADDED MISSING TRIPLES
# =========================
MISSING_COLS = ["chapter", "subject", "relation", "target", "description"]
KEY_COLS = ["chapter", "subject", "relation", "target", "description"]


def missing_to_df(records: list, reviewer_tag: str) -> pd.DataFrame:
    records = records or []
    if len(records) == 0:
        df = pd.DataFrame(columns=MISSING_COLS)
    else:
        df = pd.DataFrame(records)

    for col in MISSING_COLS:
        if col not in df.columns:
            df[col] = ""

    # Normalize key text fields for duplicate matching consistency.
    for col in ["subject", "relation", "target"]:
        df[col] = df[col].fillna("").astype(str).str.strip().str.lower()

    df["chapter"] = df["chapter"].fillna("").astype(str).str.strip()
    df["description"] = df["description"].fillna("").astype(str).str.strip()

    df = df[MISSING_COLS].copy()
    df["reviewer"] = reviewer_tag
    df["source"] = "manual_addition"
    return df


df_missing_r1 = missing_to_df(data_r1.get("missingTriples", []), "r1")
df_missing_r2 = missing_to_df(data_r2.get("missingTriples", []), "r2")

# Missing triples added by BOTH reviewers (exact match by normalized key columns).
df_missing_agreed = (
    df_missing_r1[KEY_COLS]
    .drop_duplicates()
    .merge(df_missing_r2[KEY_COLS].drop_duplicates(), on=KEY_COLS, how="inner")
    .drop_duplicates()
)
df_missing_agreed["triple_id"] = pd.NA
df_missing_agreed["r1_rating"] = "missing"
df_missing_agreed["r2_rating"] = "missing"
df_missing_agreed["final_consensus_label"] = "missing"
df_missing_agreed["reasoning"] = "Auto-agreed: both reviewers added the same missing triple."
df_missing_agreed["source"] = "manual_addition"
df_missing_agreed["bucket"] = "missing_agreed"

# Missing triples added by only ONE reviewer.
agreed_key_set = set(tuple(x) for x in df_missing_agreed[KEY_COLS].to_numpy())


def is_agreed_key(row: pd.Series) -> bool:
    return tuple(row[c] for c in KEY_COLS) in agreed_key_set


df_missing_r1_only = df_missing_r1[~df_missing_r1.apply(is_agreed_key, axis=1)].copy()
df_missing_r2_only = df_missing_r2[~df_missing_r2.apply(is_agreed_key, axis=1)].copy()

df_missing_disagreed = pd.concat([df_missing_r1_only, df_missing_r2_only], ignore_index=True)

print("df_missing_r1 shape:", df_missing_r1.shape)
print("df_missing_r2 shape:", df_missing_r2.shape)
print("df_missing_agreed shape:", df_missing_agreed.shape)
print("df_missing_disagreed shape:", df_missing_disagreed.shape)

df_missing_r1 shape: (0, 7)
df_missing_r2 shape: (29, 7)
df_missing_agreed shape: (0, 12)
df_missing_disagreed shape: (29, 7)


In [8]:
# =========================
# 4) PREPARE FOR LLM JUDGE BATCH PROCESSING
# =========================
def build_disagreed_prompt(row: pd.Series) -> str:
    ebook_context = get_ebook_context(row)
    return (
        "You are an expert KG validator for high-school chemistry curriculum.\n"
        "Decide final label for this disagreement:\n"
        f"- triple_id: {row['triple_id']}\n"
        f"- chapter: {row.get('chapter', '')}\n"
        f"- subject: {row.get('subject', '')}\n"
        f"- relation: {row.get('relation', '')}\n"
        f"- target: {row.get('target', '')}\n"
        f"- description: {row.get('description', '')}\n"
        f"- reviewer_1_label: {row['r1_rating']}\n"
        f"- reviewer_2_label: {row['r2_rating']}\n"
        f"\nRelevant ebook context:\n{ebook_context or 'No relevant ebook context found.'}\n"
        "Use the ebook context as supporting evidence when it is relevant. Do not invent facts beyond the triple and context.\n"
        "Return strict JSON with keys: reasoning, final_consensus_label.\n"
        "Use one of these labels: correct, wrong, partial, missing, abaikan.\n"
        "If final_consensus_label is partial or missing, write reasoning in Bahasa Indonesia as a clean KG relation description that can replace or enrich the original description.\n"
        "For other labels, write a concise validation rationale in Bahasa Indonesia."
    )


def build_missing_prompt(row: pd.Series) -> str:
    ebook_context = get_ebook_context(row)
    return (
        "You are validating a missing triple proposed by only one reviewer.\n"
        "Decide if this triple should be accepted as curriculum-required 'missing' "
        "or ignored as 'Abaikan'.\n"
        f"- reviewer_source: {row.get('reviewer', 'unknown')}\n"
        f"- chapter: {row.get('chapter', '')}\n"
        f"- subject: {row.get('subject', '')}\n"
        f"- relation: {row.get('relation', '')}\n"
        f"- target: {row.get('target', '')}\n"
        f"- description: {row.get('description', '')}\n"
        f"\nRelevant ebook context:\n{ebook_context or 'No relevant ebook context found.'}\n"
        "Use the ebook context as supporting evidence when it is relevant. Do not invent facts beyond the triple and context.\n"
        "Return strict JSON with keys: reasoning, final_consensus_label.\n"
        "Use one of these labels: missing, abaikan.\n"
        "If final_consensus_label is missing, write reasoning in Bahasa Indonesia as a clean KG relation description that can be used directly as the description.\n"
        "If final_consensus_label is abaikan, write a concise validation rationale in Bahasa Indonesia."
    )


GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_CLIENT = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-pro-preview")
GEMINI_MODEL_ALIASES = {}
JUDGE_CHUNK_SIZE = int(os.getenv("JUDGE_CHUNK_SIZE", "5"))
JUDGE_MAX_CHUNKS = os.getenv("JUDGE_MAX_CHUNKS", "1")
RESUME_LLM_JUDGE = os.getenv("RESUME_LLM_JUDGE", "1").strip().lower() not in {"0", "false", "no"}
EBOOK_CONTEXT_MAX_CHUNKS = int(os.getenv("EBOOK_CONTEXT_MAX_CHUNKS", "3"))
EBOOK_CONTEXT_MAX_CHARS = int(os.getenv("EBOOK_CONTEXT_MAX_CHARS", "1800"))
EBOOK_CONTEXT_MIN_SCORE = int(os.getenv("EBOOK_CONTEXT_MIN_SCORE", "2"))


def _parse_max_chunks(value: str | None) -> int | None:
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    try:
        parsed = int(text)
    except ValueError:
        return None
    return None if parsed <= 0 else parsed


JUDGE_MAX_CHUNKS = _parse_max_chunks(JUDGE_MAX_CHUNKS)


CONTEXT_STOPWORDS = {
    "yang", "dan", "atau", "dengan", "untuk", "pada", "dari", "dalam", "adalah",
    "kelas", "xii", "chapter", "subject", "relation", "target", "description",
    "memiliki", "sifat", "diformulasikan", "sebagai", "mengatur",
}


def find_ebook_context_path() -> Path | None:
    if not EBOOK_CONTEXT_DIR.exists():
        return None
    patterns = [
        f"{SUBJECT} Kelas {GRADE}_ebook_context_*.json",
        f"{SUBJECT} Kelas {GRADE}*.json",
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(EBOOK_CONTEXT_DIR.glob(pattern))
    candidates = sorted(set(candidates), key=lambda path: path.stat().st_mtime, reverse=True)
    return candidates[0] if candidates else None


def flatten_ebook_context(path: Path | None) -> list[dict]:
    if path is None:
        print(f"No ebook context file found for {SUBJECT} Kelas {GRADE}; LLM judge will run without ebook snippets.")
        return []
    data = load_json(path)
    chunks = []
    for chapter in data.get("chapters", []):
        chapter_name = chapter.get("chapter", "")
        for chunk in chapter.get("chunks", []):
            text = str(chunk.get("text", "") or "").strip()
            if not text:
                continue
            chunks.append({
                "chapter": chapter_name,
                "page": chunk.get("page_label") or chunk.get("page", "?"),
                "text": text,
            })
    print(f"Loaded ebook context: {path.name} ({len(chunks)} chunks)")
    return chunks


def tokenize_for_context(value) -> set[str]:
    tokens = re.findall(r"[0-9A-Za-zÀ-ÖØ-öø-ÿ]+", str(value or "").lower())
    return {token for token in tokens if len(token) > 2 and token not in CONTEXT_STOPWORDS}


def score_ebook_chunk(chunk: dict, query_terms: set[str], chapter_name: str) -> int:
    text = f"{chunk.get('chapter', '')} {chunk.get('text', '')}".lower()
    score = sum(1 for term in query_terms if term in text)
    if chapter_name and str(chunk.get("chapter", "")).strip().lower() == chapter_name:
        score += 3
    return score


def get_ebook_context(row: pd.Series) -> str:
    if not EBOOK_CONTEXT_CHUNKS:
        return ""
    query_fields = [
        row.get("chapter", ""),
        row.get("subject", ""),
        row.get("relation", ""),
        row.get("target", ""),
        row.get("description", ""),
    ]
    query_terms = set().union(*(tokenize_for_context(value) for value in query_fields))
    chapter_name = str(row.get("chapter", "") or "").strip().lower()
    scored = [
        (score_ebook_chunk(chunk, query_terms, chapter_name), chunk)
        for chunk in EBOOK_CONTEXT_CHUNKS
    ]
    scored = [(score, chunk) for score, chunk in scored if score >= EBOOK_CONTEXT_MIN_SCORE]
    scored.sort(key=lambda item: item[0], reverse=True)

    snippets = []
    total_chars = 0
    for score, chunk in scored[:EBOOK_CONTEXT_MAX_CHUNKS]:
        snippet = f"[{chunk.get('chapter', '')}, p.{chunk.get('page', '?')}, score={score}] {chunk.get('text', '')}"
        if total_chars + len(snippet) > EBOOK_CONTEXT_MAX_CHARS:
            remaining = EBOOK_CONTEXT_MAX_CHARS - total_chars
            if remaining > 200:
                snippets.append(snippet[:remaining].rstrip() + "...")
            break
        snippets.append(snippet)
        total_chars += len(snippet)
    return "\n\n".join(snippets)


EBOOK_CONTEXT_PATH = find_ebook_context_path()
EBOOK_CONTEXT_CHUNKS = flatten_ebook_context(EBOOK_CONTEXT_PATH)


def build_gemini_generation_config(force_json: bool = False):
    if force_json:
        return gtypes.GenerateContentConfig(response_mime_type="application/json")
    return None


def _print_llm_request(label: str, model: str, prompt: str, generation_config) -> None:
    print(f"\n=== LLM REQUEST ({label}) ===")
    print("Model:", model)
    print("Prompt:\n", prompt)
    if generation_config:
        print("Generation config:\n", generation_config.model_dump_json(indent=2, exclude_none=True))


def call_gemini(prompt: str, model: str | None = None, force_json: bool = False) -> str:
    if not GEMINI_API_KEY:
        print("Missing GEMINI_API_KEY. Set it to enable LLM calls.")
        return ""
    model = model or GEMINI_MODEL
    if model in GEMINI_MODEL_ALIASES:
        fallback_model = GEMINI_MODEL_ALIASES[model]
        print(f"Model {model!r} is not available for generateContent; using {fallback_model!r} instead.")
        model = fallback_model
    generation_config = build_gemini_generation_config(force_json=force_json)
    _print_llm_request("judge", model, prompt, generation_config)
    try:
        response = GEMINI_CLIENT.models.generate_content(
            model=model,
            contents=prompt,
            config=generation_config,
        )
    except Exception as exc:
        print(f"LLM call failed: {exc}")
        return ""
    return (getattr(response, "text", "") or "").strip()


def call_gemini_json(prompt: str, model: str | None = None) -> dict:
    text = call_gemini(prompt, model=model, force_json=True)
    if not text:
        return {}
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print("LLM response was not valid JSON.")
        return {}


def evaluate_with_llm(prompt: str) -> dict:
    response = call_gemini_json(prompt)
    if not response:
        return {"reasoning": "", "final_consensus_label": "unknown"}
    return response


def json_safe_value(value):
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value


def make_checkpoint_key(row: pd.Series, key_cols: list[str]) -> str:
    values = [str(json_safe_value(row.get(col, "")) or "").strip().lower() for col in key_cols]
    return "|".join(values)


def load_llm_checkpoint(path: Path) -> dict:
    if not RESUME_LLM_JUDGE or not path.exists():
        return {}
    checkpoint = {}
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                print(f"Skipping invalid checkpoint line {line_no} in {path.name}")
                continue
            key = record.get("checkpoint_key")
            if key:
                checkpoint[key] = record
    print(f"Loaded {len(checkpoint)} checkpoint row(s) from {path.name}")
    return checkpoint


def append_llm_checkpoint(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def checkpoint_record_to_response(record: dict) -> dict:
    return {
        "reasoning": str(record.get("reasoning", "") or ""),
        "final_consensus_label": normalize_label(record.get("final_consensus_label", "unknown")),
    }


def judge_dataframe_with_llm_checkpointed(
    df: pd.DataFrame,
    prompt_col: str,
    checkpoint_name: str,
    key_cols: list[str],
    chunk_size: int,
    max_chunks: int | None = None,
) -> pd.DataFrame:
    if df.empty:
        out = df.copy()
        out["reasoning"] = pd.Series(dtype="string")
        out["final_consensus_label"] = pd.Series(dtype="string")
        return out

    checkpoint_path = CHECKPOINT_DIR / f"{SUBJECT_SLUG}_{checkpoint_name}.jsonl"
    checkpoint = load_llm_checkpoint(checkpoint_path)
    chunk_size = max(int(chunk_size), 1)
    max_rows = len(df) if max_chunks is None else min(len(df), chunk_size * max_chunks)

    rows = []
    for row_idx, (_, row) in enumerate(df.reset_index(drop=True).iterrows()):
        row_out = row.copy()
        checkpoint_key = make_checkpoint_key(row, key_cols)
        saved = checkpoint.get(checkpoint_key)

        if saved:
            response = checkpoint_record_to_response(saved)
            print(f"Checkpoint hit [{checkpoint_name}] row={row_idx + 1}/{len(df)} key={checkpoint_key}")
        elif row_idx < max_rows:
            response = evaluate_with_llm(row[prompt_col])
            response["final_consensus_label"] = normalize_label(response.get("final_consensus_label", "unknown"))
            response["reasoning"] = str(response.get("reasoning", "") or "")
            record = {
                "checkpoint_key": checkpoint_key,
                "checkpoint_name": checkpoint_name,
                "subject": SUBJECT,
                "grade": GRADE,
                "row": {col: json_safe_value(row.get(col)) for col in df.columns if col != prompt_col},
                "reasoning": response["reasoning"],
                "final_consensus_label": response["final_consensus_label"],
            }
            append_llm_checkpoint(checkpoint_path, record)
            checkpoint[checkpoint_key] = record
            print(f"Checkpoint saved [{checkpoint_name}] row={row_idx + 1}/{len(df)} key={checkpoint_key}")
        else:
            response = {"reasoning": "", "final_consensus_label": "unknown"}

        row_out["reasoning"] = response.get("reasoning", "")
        row_out["final_consensus_label"] = normalize_label(response.get("final_consensus_label", "unknown"))
        rows.append(row_out)

    out = pd.DataFrame(rows).reset_index(drop=True)
    out["reasoning"] = out["reasoning"].fillna("").astype(str)
    out["final_consensus_label"] = out["final_consensus_label"].apply(normalize_label)
    return out


# LLM batch for rating disagreements.
df_disagreed_for_judge = df_disagreed.copy()
df_disagreed_for_judge["llm_prompt"] = df_disagreed_for_judge.apply(build_disagreed_prompt, axis=1)
df_disagreed_judged = judge_dataframe_with_llm_checkpointed(
    df_disagreed_for_judge,
    "llm_prompt",
    checkpoint_name="ratings_disagreed_llm",
    key_cols=["triple_id"],
    chunk_size=JUDGE_CHUNK_SIZE,
    max_chunks=JUDGE_MAX_CHUNKS,
)
df_disagreed_judged["source"] = "system_generated"
df_disagreed_judged["bucket"] = "ratings_disagreed_llm"

# LLM batch for one-reviewer missing additions.
df_missing_disagreed_for_judge = df_missing_disagreed.copy()
df_missing_disagreed_for_judge["llm_prompt"] = df_missing_disagreed_for_judge.apply(build_missing_prompt, axis=1)
df_missing_disagreed_judged = judge_dataframe_with_llm_checkpointed(
    df_missing_disagreed_for_judge,
    "llm_prompt",
    checkpoint_name="missing_disagreed_llm",
    key_cols=["reviewer", "chapter", "subject", "relation", "target", "description"],
    chunk_size=JUDGE_CHUNK_SIZE,
    max_chunks=JUDGE_MAX_CHUNKS,
)

if not df_missing_disagreed_judged.empty:
    df_missing_disagreed_judged["triple_id"] = pd.NA
    df_missing_disagreed_judged["r1_rating"] = df_missing_disagreed_judged["reviewer"].map(
        lambda x: "missing" if x == "r1" else pd.NA
    )
    df_missing_disagreed_judged["r2_rating"] = df_missing_disagreed_judged["reviewer"].map(
        lambda x: "missing" if x == "r2" else pd.NA
    )

df_missing_disagreed_judged["source"] = "manual_addition"
df_missing_disagreed_judged["bucket"] = "missing_disagreed_llm"


Loaded ebook context: Kimia Kelas XII_ebook_context_20260530_132659.json (235 chunks)
Loaded 19 checkpoint row(s) from kimia_ratings_disagreed_llm.jsonl
Checkpoint hit [ratings_disagreed_llm] row=1/58 key=4
Checkpoint hit [ratings_disagreed_llm] row=2/58 key=5
Checkpoint hit [ratings_disagreed_llm] row=3/58 key=6
Checkpoint hit [ratings_disagreed_llm] row=4/58 key=8
Checkpoint hit [ratings_disagreed_llm] row=5/58 key=9
Checkpoint hit [ratings_disagreed_llm] row=6/58 key=10
Checkpoint hit [ratings_disagreed_llm] row=7/58 key=11
Checkpoint hit [ratings_disagreed_llm] row=8/58 key=12
Checkpoint hit [ratings_disagreed_llm] row=9/58 key=13
Checkpoint hit [ratings_disagreed_llm] row=10/58 key=16
Checkpoint hit [ratings_disagreed_llm] row=11/58 key=17
Checkpoint hit [ratings_disagreed_llm] row=12/58 key=18
Checkpoint hit [ratings_disagreed_llm] row=13/58 key=20
Checkpoint hit [ratings_disagreed_llm] row=14/58 key=22
Checkpoint hit [ratings_disagreed_llm] row=15/58 key=25
Checkpoint hit [ratin

In [9]:
# =========================
# 5) RECOMBINE & EXPORT TO JSON
# =========================
REVIEWER_MAP = {
    "r1_rating": f"expert-{SUBJECT_SLUG}-4",
    "r2_rating": f"expert-{SUBJECT_SLUG}-6",
}


def normalize_comment_map(comment_map: dict) -> dict:
    if not isinstance(comment_map, dict):
        return {}
    normalized = {}
    for key, value in comment_map.items():
        key_str = str(key).strip()
        if key_str:
            normalized[key_str] = value
    return normalized


REVIEWER_COMMENT_MAP = {
    REVIEWER_MAP["r1_rating"]: normalize_comment_map(data_r1.get("comments", {})),
    REVIEWER_MAP["r2_rating"]: normalize_comment_map(data_r2.get("comments", {})),
}


def lookup_comment(comment_map: dict, triple_id):
    if not comment_map or triple_id is None or pd.isna(triple_id):
        return None
    if isinstance(triple_id, float) and triple_id.is_integer():
        triple_key = str(int(triple_id))
    else:
        triple_key = str(triple_id).strip()
    if triple_key in comment_map:
        return comment_map.get(triple_key)
    if triple_key.isdigit():
        return comment_map.get(str(int(triple_key)))
    return None


def build_refined_description(row: pd.Series) -> str:
    base = str(row.get("description", "") or "").strip()
    label = normalize_label(row.get("final_consensus_label", "unknown"))
    reasoning = str(row.get("reasoning", "") or "").strip()
    source = str(row.get("source", "") or "").strip()

    # Only adopt LLM-judge reasoning as the description for genuinely refined
    # (partial/missing) triples. Never adopt the auto-agreed sentinel, which is
    # a pipeline status note, not a description.
    auto_sentinel = "Auto-agreed: both reviewers gave the same label."
    if (
        label in {"partial", "missing"}
        and reasoning
        and reasoning != auto_sentinel
        and source != "system_generated"
    ):
        return reasoning
    return base


def get_refined_description(row: pd.Series) -> str:
    refined = row.get("refined_description", pd.NA)
    if pd.notna(refined) and str(refined).strip() != "":
        return str(refined).strip()
    return str(row.get("description", "") or "").strip()


def get_added_by_reviewers(row: pd.Series) -> list[str]:
    if str(row.get("source", "")).strip() != "manual_addition":
        return []

    reviewer_tag = str(row.get("reviewer", "") or "").strip().lower()
    if reviewer_tag == "r1":
        return [REVIEWER_MAP["r1_rating"]]
    if reviewer_tag == "r2":
        return [REVIEWER_MAP["r2_rating"]]

    added_by = []
    if normalize_label(row.get("r1_rating", "")) == "missing":
        added_by.append(REVIEWER_MAP["r1_rating"])
    if normalize_label(row.get("r2_rating", "")) == "missing":
        added_by.append(REVIEWER_MAP["r2_rating"])
    return added_by


final_cols = [
    "triple_id",
    "chapter",
    "subject",
    "relation",
    "target",
    "description",
    "refined_description",
    "r1_rating",
    "r2_rating",
    "final_consensus_label",
    "reasoning",
    "added_by_reviewers",
    "source",
    "bucket",
]

# Mark which reviewer(s) proposed manually added triples.
for df in [df_agreed, df_disagreed_judged, df_missing_agreed, df_missing_disagreed_judged]:
    df["added_by_reviewers"] = df.apply(get_added_by_reviewers, axis=1)

# Ensure compatible schema before concat.
for df in [df_agreed, df_disagreed_judged, df_missing_agreed, df_missing_disagreed_judged]:
    for col in final_cols:
        if col not in df.columns:
            df[col] = pd.NA

# Clean labels across all pieces.
df_agreed["final_consensus_label"] = df_agreed["final_consensus_label"].apply(normalize_label)
df_disagreed_judged["final_consensus_label"] = df_disagreed_judged["final_consensus_label"].apply(normalize_label)
df_missing_agreed["final_consensus_label"] = df_missing_agreed["final_consensus_label"].apply(normalize_label)
df_missing_disagreed_judged["final_consensus_label"] = df_missing_disagreed_judged["final_consensus_label"].apply(normalize_label)

df_gold_standard = pd.concat(
    [
        df_agreed[final_cols],
        df_disagreed_judged[final_cols],
        df_missing_agreed[final_cols],
        df_missing_disagreed_judged[final_cols],
    ],
    ignore_index=True,
)

# Optional stable ordering: numeric triple_id first, manual additions at end.
df_gold_standard["_sort_key"] = pd.to_numeric(df_gold_standard["triple_id"], errors="coerce")
df_gold_standard = df_gold_standard.sort_values(
    by=["_sort_key", "source", "chapter", "subject", "relation", "target"],
    na_position="last",
).drop(columns=["_sort_key"]).reset_index(drop=True)

# Refine descriptions using expert comments for partial/missing labels.
df_gold_standard["refined_description"] = df_gold_standard.apply(
    build_refined_description, axis=1
)

# Export flat gold standard as requested.
df_gold_standard.to_json(
    OUTPUT_PATH,
    orient="records",
    indent=2,
    force_ascii=False,
)

print("Export complete:", display_path(OUTPUT_PATH))
print("df_gold_standard shape:", df_gold_standard.shape)
display(df_gold_standard.head(10))

Export complete: kimia_gold_standard.json
df_gold_standard shape: (195, 14)


,triple_id,chapter,subject,relation,target,description,refined_description,r1_rating,r2_rating,final_consensus_label,reasoning,added_by_reviewers,source,bucket
0,0,LARUTAN DAN KOLOID,Asam Brønsted-Lowry,MEMILIKI_SIFAT,Donor Proton,Asam Brønsted-Lowry dicirikan oleh kemampuanny...,Asam Brønsted-Lowry dicirikan oleh kemampuanny...,correct,correct,correct,Auto-agreed: both reviewers gave the same label.,[],system_generated,ratings_agreed
1,1,LARUTAN DAN KOLOID,Basa Brønsted-Lowry,MEMILIKI_SIFAT,Akseptor Proton,Basa Brønsted-Lowry dicirikan oleh kemampuanny...,Basa Brønsted-Lowry dicirikan oleh kemampuanny...,correct,correct,correct,Auto-agreed: both reviewers gave the same label.,[],system_generated,ratings_agreed
2,2,LARUTAN DAN KOLOID,Asam Lewis,MEMILIKI_SIFAT,Akseptor Pasangan Elektron Bebas,Asam Lewis berperan sebagai penerima pasangan ...,Asam Lewis berperan sebagai penerima pasangan ...,correct,correct,correct,Auto-agreed: both reviewers gave the same label.,[],system_generated,ratings_agreed
3,3,LARUTAN DAN KOLOID,Basa Lewis,MEMILIKI_SIFAT,Donor Pasangan Elektron Bebas,Basa Lewis berperan sebagai pemberi pasangan e...,Basa Lewis berperan sebagai pemberi pasangan e...,correct,correct,correct,Auto-agreed: both reviewers gave the same label.,[],system_generated,ratings_agreed
4,4,LARUTAN DAN KOLOID,pH,DIFORMULASIKAN_SEBAGAI,–log [H+],pH dihitung dari nilai negatif logaritma konse...,pH dihitung dari nilai negatif logaritma konse...,correct,missing,correct,Teks menyebutkan secara eksplisit bahwa pH dir...,[],system_generated,ratings_disagreed_llm
5,5,LARUTAN DAN KOLOID,pH,MENGATUR,Keasaman Larutan,Nilai pH menentukan tingkat keasaman atau keba...,Nilai pH menentukan tingkat keasaman atau keba...,wrong,missing,wrong,Relasi MENGATUR tidak tepat. Nilai pH tidak be...,[],system_generated,ratings_disagreed_llm
6,6,LARUTAN DAN KOLOID,Tetapan Swaionisasi Air (Kw),DIFORMULASIKAN_SEBAGAI,[H3O+][OH–],Kw adalah hasil kali konsentrasi ion hidronium...,Kw adalah hasil kali konsentrasi ion hidronium...,correct,missing,correct,Konteks secara eksplisit menyatakan bahwa teta...,[],system_generated,ratings_disagreed_llm
7,7,LARUTAN DAN KOLOID,Tetapan Swaionisasi Air (Kw),BERGANTUNG_PADA,Suhu,Nilai Kw bervariasi tergantung pada temperatur...,Nilai Kw bervariasi tergantung pada temperatur...,correct,correct,correct,Auto-agreed: both reviewers gave the same label.,[],system_generated,ratings_agreed
8,8,LARUTAN DAN KOLOID,Asam Lemah,DIFORMULASIKAN_SEBAGAI,Ka,Kekuatan asam lemah diukur dengan tetapan kese...,Kekuatan asam lemah diukur dengan tetapan kese...,wrong,correct,wrong,Relasi DIFORMULASIKAN_SEBAGAI tidak tepat kare...,[],system_generated,ratings_disagreed_llm
9,9,LARUTAN DAN KOLOID,Asam Lemah,DIFORMULASIKAN_SEBAGAI,α (Derajat Ionisasi),Derajat ionisasi menunjukkan seberapa banyak a...,Derajat ionisasi menunjukkan seberapa banyak a...,wrong,correct,wrong,Asam lemah tidak 'diformulasikan sebagai' dera...,[],system_generated,ratings_disagreed_llm


In [10]:
# =========================
# 6) CONVERT GOLD STANDARD TO KG WITH CONSENSUS
# =========================
def build_review_payload(row: pd.Series, triple_id=None) -> dict:
    ratings = {}
    comments = {}
    for col_name, reviewer in REVIEWER_MAP.items():
        value = row.get(col_name, pd.NA)
        if pd.notna(value) and str(value).strip() != "":
            ratings[reviewer] = str(value)
        comment_value = lookup_comment(REVIEWER_COMMENT_MAP.get(reviewer, {}), triple_id)
        if comment_value is not None and str(comment_value).strip() != "":
            comments[reviewer] = str(comment_value)

    consensus = normalize_label(row.get("final_consensus_label", "unknown"))
    review_payload = {
        "status": consensus,
        "consensus": consensus,
        "n_reviewers": len([
            v for v in ratings.values() if v is not None and str(v).strip() != ""
        ]),
        "ratings": ratings,
        "comments": comments,
    }
    added_by_reviewers = row.get("added_by_reviewers", [])
    if isinstance(added_by_reviewers, list) and added_by_reviewers:
        review_payload["added_by_reviewers"] = added_by_reviewers
    return review_payload


def merge_expert_review(existing: dict, new_review: dict) -> dict:
    existing = existing if isinstance(existing, dict) else {}
    ratings = existing.get("ratings", {})
    if not isinstance(ratings, dict):
        ratings = {}
    comments = existing.get("comments", {})
    if not isinstance(comments, dict):
        comments = {}

    merged_ratings = dict(ratings)
    merged_ratings.update(new_review.get("ratings", {}))

    merged_comments = dict(comments)
    for key, value in (new_review.get("comments", {}) or {}).items():
        if key not in merged_comments:
            merged_comments[key] = value

    merged = dict(existing)
    merged["status"] = new_review.get("status", merged.get("status"))
    merged["consensus"] = new_review.get("consensus", merged.get("consensus"))
    merged["ratings"] = merged_ratings
    merged["comments"] = merged_comments
    merged["n_reviewers"] = len([
        v for v in merged_ratings.values() if v is not None and str(v).strip() != ""
    ])
    return merged


def iter_kg_relation_nodes(kg_json):
    """Yield (source_concept_name, relation_node) over concept relations.

    chapter_relations are intentionally excluded to match the reviewer-id walk.
    """
    for chapter_node in kg_json.get("chapters", []):
        for subtopic_node in chapter_node.get("subtopics", []):
            for concept_node in subtopic_node.get("concepts", []):
                for relation_node in concept_node.get("relations", []):
                    yield concept_node.get("name", ""), relation_node


# Content-key index: join consensus payloads onto the matching triple by
# (source concept, relation type, target) rather than by fragile positional id.
relation_index = {}
for _concept_name, _relation_node in iter_kg_relation_nodes(kg_json_data):
    _key = _content_key(
        _concept_name,
        _relation_node.get("type", ""),
        _relation_node.get("target_concept", _relation_node.get("target", "")),
    )
    relation_index.setdefault(_key, _relation_node)

# Update existing relations with consensus payloads.
comment_hits = 0
for _, row in df_gold_standard[pd.notna(df_gold_standard["triple_id"])].iterrows():
    triple_id = coerce_triple_id(row["triple_id"])
    relation_node = relation_index.get(
        _content_key(row.get("subject"), row.get("relation"), row.get("target"))
    )
    if relation_node is None:
        continue
    review_payload = build_review_payload(row, triple_id)
    if review_payload.get("comments"):
        comment_hits += len(review_payload["comments"])
    relation_node["expert_review"] = merge_expert_review(
        relation_node.get("expert_review"), review_payload
    )
    refined_description = get_refined_description(row)
    if refined_description:
        relation_node["description"] = refined_description

# Add manual missing triples (skip consensus=abaikan/unknown).
def find_chapter_node(kg_json, chapter_name: str):
    chapter_key = chapter_name.strip().lower()
    for chapter_node in kg_json.get("chapters", []):
        if str(chapter_node.get("chapter", "")).strip().lower() == chapter_key:
            return chapter_node
    return None


def find_or_create_concept(chapter_node, subject_name: str):
    subject_key = subject_name.strip().lower()
    for subtopic_node in chapter_node.get("subtopics", []):
        for concept_node in subtopic_node.get("concepts", []):
            if str(concept_node.get("name", "")).strip().lower() == subject_key:
                return concept_node

    subtopics = chapter_node.get("subtopics", [])
    expert_subtopic = None
    for subtopic_node in subtopics:
        if str(subtopic_node.get("name", "")).strip().lower() == "expert proposed triples":
            expert_subtopic = subtopic_node
            break
    if expert_subtopic is None:
        expert_subtopic = {"name": "Expert Proposed Triples", "concepts": []}
        subtopics.append(expert_subtopic)
        chapter_node["subtopics"] = subtopics

    new_concept = {
        "name": subject_name.strip(),
        "description": "",
        "glossary_validated": False,
        "materi_pokok_ref": "",
        "relations": [],
        "provenance": "expert-added",
    }
    expert_subtopic["concepts"].append(new_concept)
    return new_concept


def relation_exists(concept_node, relation_type: str, target: str, description: str) -> bool:
    relation_key = relation_type.strip().lower()
    target_key = target.strip().lower()
    desc_key = description.strip().lower()
    for relation_node in concept_node.get("relations", []):
        rel_type = str(relation_node.get("type", "")).strip().lower()
        rel_target = str(
            relation_node.get("target", relation_node.get("target_concept", ""))
        ).strip().lower()
        rel_desc = str(relation_node.get("description", "")).strip().lower()
        if rel_type == relation_key and rel_target == target_key and rel_desc == desc_key:
            return True
    return False


missing_rows = df_gold_standard[df_gold_standard["triple_id"].isna()].copy()
missing_rows = missing_rows[
    missing_rows["final_consensus_label"].apply(
        lambda x: normalize_label(x) not in {"abaikan", "unknown"}
    )
]

for _, row in missing_rows.iterrows():
    chapter_name = str(row.get("chapter", "")).strip()
    subject_name = str(row.get("subject", "")).strip()
    relation_type = str(row.get("relation", "")).strip()
    target = str(row.get("target", "")).strip()
    description = get_refined_description(row)

    if not chapter_name or not subject_name or not relation_type or not target:
        continue

    chapter_node = find_chapter_node(kg_json_data, chapter_name)
    if chapter_node is None:
        continue

    concept_node = find_or_create_concept(chapter_node, subject_name)
    if relation_exists(concept_node, relation_type, target, description):
        continue

    review_payload = build_review_payload(row)
    new_relation = {
        "type": relation_type,
        "target": target,
        "description": description,
        "provenance": "expert-added",
        "added_by_reviewers": review_payload.get("added_by_reviewers", []),
        "expert_review": review_payload,
    }
    concept_node.setdefault("relations", []).append(new_relation)

# Export KG with consensus fields embedded in relations.
with OUTPUT_KG_PATH.open("w", encoding="utf-8") as f:
    json.dump(kg_json_data, f, ensure_ascii=False, indent=2)

print("Consensus KG export complete:", display_path(OUTPUT_KG_PATH))
print("Reviewer comments attached:", comment_hits)


Consensus KG export complete: kg/Kimia Kelas XII.consensus.json
Reviewer comments attached: 62
